# Power and design

The power calculation for the primary metric, worked through step by step. The baseline conversion rate used below is pulled directly from `00_eda_pre_registration.ipynb`, which validates it (and other design assumptions) against a pooled, non-by-arm look at the data, before any outcome is compared across arms.

## Locked inputs

- Primary metric: conversion rate
- MDE: 0.5 percentage points absolute
- Alpha: 0.05 (two-sided), Power: 0.90
- Primary family: 3 comparisons (Mens vs No-Email, Womens vs No-Email, Mens vs Womens), Holm correction
- Planning alpha for sample size: alpha / 3 (the conservative, Bonferroni-equivalent bar Holm's smallest p-value must clear), so the power claim stays honest after correction is applied

Baseline conversion rate is reloaded and recomputed here (pooled across all arms, matches the 0.9031% validated in `00_eda_pre_registration.ipynb`), so this notebook is self-contained.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from scipy import stats
from load import load_hillstrom

df = load_hillstrom()

baseline_conversion = df["conversion"].mean()
mde_absolute = 0.005
alpha = 0.05
power = 0.90
n_primary_comparisons = 3
planning_alpha = alpha / n_primary_comparisons

print(f"baseline conversion rate (p1): {baseline_conversion:.4%}")
print(f"MDE (absolute): {mde_absolute:.4%}")
print(f"target power: {power:.0%}")
print(f"two-sided alpha (uncorrected): {alpha}")
print(f"planning alpha (alpha / {n_primary_comparisons}): {planning_alpha:.5f}")

baseline conversion rate (p1): 0.9031%
MDE (absolute): 0.5000%
target power: 90%
two-sided alpha (uncorrected): 0.05
planning alpha (alpha / 3): 0.01667


## Required sample size per arm

For comparing two proportions (p1 = baseline rate, p2 = baseline + MDE), the standard sample size formula per arm is:

n = (z_(alpha/2) + z_power)^2 * (p1*(1-p1) + p2*(1-p2)) / (p1 - p2)^2

z_(alpha/2) is the critical value that cuts off alpha/2 in each tail of a standard normal distribution (how extreme a result needs to be to count as significant). z_power is the critical value for the target power (how far p2 needs to sit from p1 for the test to reliably detect it). Computing both first, as their own numbers, before combining them.

In [2]:
z_alpha_half = stats.norm.ppf(1 - planning_alpha / 2)
z_power = stats.norm.ppf(power)

print(f"z_(alpha/2), using planning alpha {planning_alpha:.5f}: {z_alpha_half:.4f}")
print(f"z_power, for {power:.0%} power: {z_power:.4f}")

z_(alpha/2), using planning alpha 0.01667: 2.3940
z_power, for 90% power: 1.2816


### The two proportions and their variance terms

p1 is the baseline conversion rate, p2 is p1 plus the MDE, the smallest lift we want to reliably detect. Each proportion contributes a variance term p*(1-p), these get summed in the numerator of the sample size formula.

In [3]:
p1 = baseline_conversion
p2 = p1 + mde_absolute

var_p1 = p1 * (1 - p1)
var_p2 = p2 * (1 - p2)
variance_sum = var_p1 + var_p2
effect_size = p2 - p1

print(f"p1 (baseline): {p1:.4%}")
print(f"p2 (baseline + MDE): {p2:.4%}")
print(f"var_p1 = p1*(1-p1): {var_p1:.6f}")
print(f"var_p2 = p2*(1-p2): {var_p2:.6f}")
print(f"variance_sum: {variance_sum:.6f}")
print(f"effect_size (p2 - p1): {effect_size:.4f}")

p1 (baseline): 0.9031%
p2 (baseline + MDE): 1.4031%
var_p1 = p1*(1-p1): 0.008950
var_p2 = p2*(1-p2): 0.013834
variance_sum: 0.022784
effect_size (p2 - p1): 0.0050


### Putting it together: required n per arm

Combining the z-scores and variance terms into the full formula, then rounding up since you can't enroll a fractional customer.

In [4]:
numerator = (z_alpha_half + z_power) ** 2 * variance_sum
denominator = effect_size ** 2
required_n_per_arm = np.ceil(numerator / denominator)

print(f"numerator: (z_alpha/2 + z_power)^2 * variance_sum = {numerator:.4f}")
print(f"denominator: effect_size^2 = {denominator:.6f}")
print(f"required n per arm: {required_n_per_arm:,.0f}")

numerator: (z_alpha/2 + z_power)^2 * variance_sum = 0.3078
denominator: effect_size^2 = 0.000025
required n per arm: 12,313


## Comparing to the actual sample

Checking the required n per arm against how many customers are actually in each arm. This only looks at `segment` counts, not any outcome column, so it doesn't cross the pre-registration line, it's the same kind of check the loader already does (arm labels exist), just counted.

In [5]:
arm_sizes = df["segment"].value_counts()
smallest_arm = arm_sizes.min()

print("actual customers per arm:")
print(arm_sizes)
print()
print(f"required n per arm: {required_n_per_arm:,.0f}")
print(f"smallest actual arm: {smallest_arm:,}")
print(f"margin: {smallest_arm / required_n_per_arm:.2f}x the requirement")

actual customers per arm:
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

required n per arm: 12,313
smallest actual arm: 21,306
margin: 1.73x the requirement
